
# VOICE ChatBOT


1. 사용자 입력을 음성으로 받는다.
2. STT를 적용하여 텍스트로 변환한다.
3. 변환된 텍스트를 입력으로 하여 프롬프트 엔지니어링을 해 api 요청을 보낸다.
4. 반환받은 응답을 TTS를 적용하여 음성으로 재생한다.
 (+) 2에서 입력된 텍스트와 4에서 반환된 응답을 채팅 내역 보듯이 (카카오톡 대화처럼) 현출되도록 출력한다.

In [57]:
pip install openai gtts SpeechRecognition pydub pandas

Note: you may need to restart the kernel to use updated packages.


In [71]:
import openai
import speech_recognition as sr
from gtts import gTTS
import os
import IPython.display as ipd
from datetime import datetime
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv()

# os.environ에 직접 대입하는 대신, env 파일에 정의된 값을 가져옵니다.
# 만약 에러가 났던 9번 줄을 수정한다면 아래와 같이 따옴표를 쓰거나 직접 문자열을 넣어야 합니다.
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [72]:
def speak(text):
    """TTS: 텍스트를 음성으로 변환하여 재생"""
    tts = gTTS(text=text, lang='ko')
    tts.save("response.mp3")
    return ipd.Audio("response.mp3", autoplay=True)

def listen():
    """STT: 마이크 입력을 텍스트로 변환"""
    r = sr.Recognizer()
    with sr.Microphone() as source:
        print("아이의 증상을 말씀해 주세요... (듣고 있습니다)")
        audio = r.listen(source)
    try:
        user_input = r.recognize_google(audio, language='ko-KR')
        return user_input
    except:
        return None

def get_medical_advice(user_text):
    """Prompt Engineering: 소아과 상담 특화 프롬프트"""
    system_prompt = """
    당신은 친절하고 전문적인 소아과 상담 AI '우리아이 케어'입니다.
    부모님의 걱정에 공감하며, 아이의 증상에 따른 응급 처치나 병원 방문 필요성을 안내하세요.
    단, 반드시 '이 답변은 참고용이며 정확한 진단은 의사에게 받으세요'라는 문구를 포함하세요.
    """
    
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_text}
        ]
    )
    return response.choices[0].message.content

In [73]:
from IPython.display import HTML, display

def display_chat(role, message):
    """카카오톡 스타일의 메시지 박스 출력"""
    time_str = datetime.now().strftime("%H:%M")
    
    if role == "user":
        bg_color = "#FEE500"  # 카톡 노란색
        float_dir = "right"
        text_align = "right"
        margin = "0 0 10px 50px"
    else:
        bg_color = "#FFFFFF"  # 흰색
        float_dir = "left"
        text_align = "left"
        margin = "0 50px 10px 0"

    chat_html = f"""
    <div style="overflow: auto; margin-bottom: 10px;">
        <div style="float: {float_dir}; background-color: {bg_color}; padding: 10px; border-radius: 10px; 
                    max-width: 70%; box-shadow: 1px 1px 2px #888; margin: {margin}; text-align: left;">
            <div style="font-size: 12px; color: #555; margin-bottom: 5px;"><b>{role.upper()}</b></div>
            <div style="font-size: 14px; line-height: 1.5;">{message}</div>
            <div style="font-size: 10px; color: #999; text-align: right; margin-top: 5px;">{time_str}</div>
        </div>
    </div>
    """
    display(HTML(chat_html))

# --- 실시간 실행 루프 ---
def run_child_care_chatbot():
    user_voice_input = listen()
    
    if user_voice_input:
        # 1. 사용자 입력 (STT 결과) 출력
        display_chat("user", user_voice_input)
        
        # 2. LLM 응답 생성
        ai_response = get_medical_advice(user_voice_input)
        
        # 3. AI 응답 출력
        display_chat("assistant", ai_response)
        
        # 4. 음성 재생 (TTS)
        return speak(ai_response)
    else:
        print("음성을 인식하지 못했습니다. 다시 시도해주세요.")

# 실행
run_child_care_chatbot()

아이의 증상을 말씀해 주세요... (듣고 있습니다)
